In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# TRAIN
train = spark.table("databricks_gsales.bronze.train")

w_train = Window.partitionBy(F.col("id").cast("bigint")).orderBy(F.col("date").cast("date"))

silver_train = (
    train
    .withColumn("id", F.col("id").cast("bigint"))
    .withColumn("date", F.col("date").cast("date"))
    .withColumn("store_nbr", F.col("store_nbr").cast("double").cast("int"))
    .withColumn("family", F.trim("family"))
    .withColumn("sales", F.col("sales").cast("double"))
    .withColumn("onpromotion", F.col("onpromotion").cast("double").cast("int"))
    .filter(
        F.col("id").isNotNull() &
        F.col("date").isNotNull() &
        F.col("store_nbr").isNotNull() &
        F.col("sales").isNotNull() & 
        (F.col("sales") >= 0)
    )
    .withColumn("rn", F.row_number().over(w_train))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

silver_train.write.mode("overwrite").saveAsTable(
    "databricks_gsales.silver.silver_train"
)

print("silver_train created successfully")
# STORES
stores = spark.table("databricks_gsales.bronze.stores")

w_stores = Window.partitionBy(
    F.col("store_nbr").cast("double").cast("int")
).orderBy(F.trim("city"))

silver_stores = (
    stores
    .withColumn("store_nbr", F.col("store_nbr").cast("double").cast("int"))
    .withColumn("city", F.trim("city"))
    .withColumn("state", F.trim("state"))
    .withColumn("type", F.trim("type"))
    .withColumn("cluster", F.col("cluster").cast("double").cast("int"))
    .filter(F.col("store_nbr").isNotNull())
    .withColumn("rn", F.row_number().over(w_stores))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

silver_stores.write.mode("overwrite").saveAsTable(
    "databricks_gsales.silver.silver_stores"
)

# TRANSACTIONS
silver_transactions = (
    spark.table("databricks_gsales.bronze.transactions")
    .withColumn("date", F.col("date").cast("date"))
    .withColumn("store_nbr", F.col("store_nbr").cast("double").cast("int"))
    .withColumn("transactions", F.col("transactions").cast("double").cast("int"))
    .filter(
        F.col("date").isNotNull() &
        F.col("store_nbr").isNotNull()
    )
)

silver_transactions.write.mode("overwrite").saveAsTable(
    "databricks_gsales.silver.silver_transactions"
)

# OIL
silver_oil = (
    spark.table("databricks_gsales.bronze.oil")
    .withColumn("date", F.col("date").cast("date"))
    .withColumn("dcoilwtico", F.col("dcoilwtico").cast("double"))
    .filter(F.col("date").isNotNull())
)

silver_oil.write.mode("overwrite").saveAsTable(
    "databricks_gsales.silver.silver_oil"
)

# HOLIDAYS
silver_holidays = (
    spark.table("databricks_gsales.bronze.holidays_events")
    .withColumn("date", F.col("date").cast("date"))
    .withColumn("type", F.trim("type"))
    .withColumn("locale", F.trim("locale"))
    .withColumn("locale_name", F.trim("locale_name"))
    .withColumn("description", F.trim("description"))
    .withColumn("transferred", F.col("transferred").cast("boolean"))
    .filter(F.col("date").isNotNull())
)

silver_holidays.write.mode("overwrite").saveAsTable(
    "databricks_gsales.silver.silver_holidays_events"
)

# TEST
test = spark.table("databricks_gsales.bronze.test")

w_test = Window.partitionBy(
    F.col("id").cast("bigint")
).orderBy(F.col("date").cast("date"))

silver_test = (
    test
    .withColumn("id", F.col("id").cast("bigint"))
    .withColumn("date", F.col("date").cast("date"))
    .withColumn("store_nbr", F.col("store_nbr").cast("double").cast("int"))
    .withColumn("family", F.trim("family"))
    .withColumn("onpromotion", F.col("onpromotion").cast("double").cast("int"))
    .filter(
        F.col("id").isNotNull() &
        F.col("date").isNotNull() &
        F.col("store_nbr").isNotNull()
    )
    .withColumn("rn", F.row_number().over(w_test))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

silver_test.write.mode("overwrite").saveAsTable(
    "databricks_gsales.silver.silver_test"
)

print("All Silver tables created successfully")

silver_train created successfully
All Silver tables created successfully
